# Example: AEPyBaMM with Molicel P45B Parameters

This example demonstrates using AEPyBaMM to solve an electrochemical simulation of the Molicel P45B cell, using About:Energy's open-source Doyle-Fuller-Newman (DFN) model parameters (see "[Physics-Based Parameterisation of a High-Power Silicon–Graphite Lithium-Ion Battery: Electrochemical Model of the Molicel P45B Cell](https://doi.org/10.1149/1945-7111/aea1c1)" for a full scientific discussion).

We load the parameters from the provided [Battery Parameter eXchange (BPX)](https://bpxstandard.com/) JSON file. We will run a constant current discharge using the PyBaMM experiment class and a dynamic eVTOL profile using an experimental current load, and compare both against experimental data taken from About:Energy's [open-source data release for the Molicel P45B cell](https://zenodo.org/records/19052626).

### Housekeeping
Please ensure all packages are installed in your virtual environment. 

- aepybamm >= 0.2.3
- pybamm == 26.7

In [ ]:
import matplotlib.pyplot as plt
import pybamm

from aepybamm import (
    compare,
    get_params,
    solve_from_expdata,
)

### Section 1: Parameter set and experimental data
First, we specify the location of the BPX parameter set and the model options. The P45B has a silicon–graphite negative electrode, so we enable a blended negative electrode (the positive electrode is a single material). We also include a one-state hysteresis model, which captures the silicon hysteresis in the negative electrode OCP.

We then specify the experimental data files: a 3C constant current discharge at 25°C and an eVTOL drive cycle at 60°C. The `COL_CSV` dictionary tells AEPyBaMM which columns hold time, current and voltage.

In [ ]:
# Parameter source information
fp_params = "params/Molicel_P45B_bpx.json"
MOLICEL_P45B_STANDARD_OPTS = {
    "model_type": "DFN",
    "hysteresis_model": "one-state",
    "blended_electrode": (True, False),  # neg, pos
}

# Experimental data
fp_discharge = "data/MOLICEL_P45B_025degC_3C_Dch.csv"
fp_evtol = "data/MOLICEL_P45B_060degC_eVTOL.csv"

# Standard CSV column layout
COL_CSV = {
    "t": 0,
    "I": 1,
    "V": 2,
}

### Section 2: Initialising the model and parameter set
We can now initialise the parameter set and model as self-consistent PyBaMM objects by using the AEPyBaMM `get_params` function. Both tests begin from a fully charged cell, so we set the initial SOC to 0.995 (= 99.5%) and the preceding hysteresis state to `"charge"`, so the hysteresis branch matches the cell's recent history.

In [ ]:
parameter_values, model = get_params(
    src=fp_params,
    **MOLICEL_P45B_STANDARD_OPTS,
    SOC_init=0.995,
    hysteresis_preceding_state="charge",
)

### Section 3: Constant current discharge
Next, we use the [PyBaMM experiment class](https://docs.pybamm.org/en/latest/source/api/experiment/index.html) to define a 3C discharge to 2.5 V at 25°C. We create a PyBaMM simulation with this experiment, and the self-consistent parameter values and model objects we retrieved from `get_params` earlier. We solve the simulation, and compare it against the experimental data using the AEPyBaMM `compare` function.

- _Note:_ `compare` plots discharge current as negative. This is opposite to PyBaMM's internal convention.  

In [ ]:
experiment_3C_Dch = pybamm.Experiment(
    [
        "Discharge at 3C until 2.5 V",
    ],
    temperature=298.15,
)

sim_discharge = pybamm.Simulation(
    model,
    parameter_values=parameter_values,
    experiment=experiment_3C_Dch,
)
sol_discharge = sim_discharge.solve()
compare(sol_discharge, fp_discharge, COL_CSV)

### Section 4: Dynamic eVTOL profile
We can also simulate experimental current loads reported as a current time series. AEPyBaMM provides the `solve_from_expdata` function to wrap PyBaMM instructions that drive the model directly from the measured current. We set the ambient temperature to 60 °C to match the test conditions, then compare the result to the experimental dat as before.

In [ ]:
sol_evtol = solve_from_expdata(
    parameter_values,
    model,
    fp_evtol,
    dict_cols=COL_CSV,
    Tamb_degC=60,
)
compare(sol_evtol, fp_evtol, COL_CSV)

### Section 5: Accessing solution variables
Any variable in the model can be read from the solution by name. As the P45B has a blended negative electrode, we can look at how lithium is shared between the graphite (primary) and silicon (secondary) phases during the eVTOL profile.

As you can see, the eVTOL profile keeps the cell above SOC 30%, so that most of the negative electrode delithiation comes from graphite, not silicon. Nonetheless, the silicon material contributes during higher power pulses.

In [ ]:
time = sol_evtol["Time [s]"].entries

Qrel_Ah = sol_evtol["Discharge capacity [A.h]"].entries
Qnom_Ah = parameter_values["Nominal cell capacity [A.h]"]
soc = 1 - Qrel_Ah / Qnom_Ah

sto_graphite = sol_evtol["Negative electrode primary stoichiometry"].entries
sto_silicon = sol_evtol["Negative electrode secondary stoichiometry"].entries

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax_soc, ax_xli = ax

ax_soc.plot(time, soc, 'k')
ax_soc.set_xlabel("Time [s]")
ax_soc.set_ylabel("Cell SOC")
ax_soc.grid()

ax_xli.plot(time, sto_graphite, label="Graphite (primary)")
ax_xli.plot(time, sto_silicon, label="Silicon (secondary)")
ax_xli.set_xlabel("Time [s]")
ax_xli.set_ylabel("Lithiation extent")
ax_xli.legend()
ax_xli.grid()
plt.show()